In [ ]:
!pip install -U torchao -q

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
import torch

In [ ]:
import zipfile

path = "/content/tinyllama-instruction.zip"

with zipfile.ZipFile(path, "r") as zip_ref:
    zip_ref.extractall("/content/")
model_path = "/content/checkpoint-3"

In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map = "auto")

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
question = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
prompt = f"### Instruction:\n{question}\n### Input:\n{question}\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.9,
    top_p=0.8,
    do_sample=True,
    repetition_penalty=1.1
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
### Input:
Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
### Response:
Artificial intelligence is increasingly being used to help speed up and simplify the drug discovery and development process, and is already having an impact on the industry. One example is the use of machine learning and artificial intelligence to identify new drugs that may have potential for treating certain diseases.
This is one example of a company using AI to find better treatments for diseases. However, AI is not limited to simply identifying new drugs; it can also be used to


In [ ]:
# !pip install -U trl bitsandbytes -q

In [ ]:
from trl import DPOTrainer
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
instruction_checkpoint = "/content/checkpoint-3"

In [ ]:
dataset = load_dataset('csv', data_files='/content/pharma_preference_data.csv')['train']

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers.utils import quantization_config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

# Load the base model without quantization first for merging purposes
model = AutoModelForCausalLM.from_pretrained(base_model, quantization_config = bnb_config, device_map='auto')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha = 16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules = ['q_proj', "v_proj"]
)

In [ ]:
# like this lora delta patch will be stacked and will cause loss unstable and model hellucination issues
# pref_model_lora = get_peft_model(instruction_model, lora_config)

In [ ]:
model = PeftModel.from_pretrained(model, instruction_checkpoint)
model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
pref_model = get_peft_model(model, lora_config)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from trl import DPOTrainer, DPOConfig

In [ ]:
dpo_config = DPOConfig(
    output_dir = './tinyllama-preference-allignment',
    learning_rate = 2e-5,
    per_device_train_batch_size = 2,
    num_train_epochs = 20,
    gradient_accumulation_steps = 8,
    beta = 0.1,
    report_to = 'none',
    logging_dir = 'none',
    logging_steps = 10,
    remove_unused_columns = False,
    loss_type = 'sigmoid'
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = DPOTrainer(
    model = pref_model,
    ref_model = None,
    train_dataset = dataset,
    processing_class = tokenizer,
    args = dpo_config
)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,0.669461
20,0.635372


TrainOutput(global_step=20, training_loss=0.6524165630340576, metrics={'train_runtime': 80.0281, 'train_samples_per_second': 1.25, 'train_steps_per_second': 0.25, 'total_flos': 98496839540736.0, 'train_loss': 0.6524165630340576, 'epoch': 20.0})

In [ ]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

In [ ]:
prompt = f"### Instruction:\n{question}\n### Input:\n{question}\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.9,
    top_p=0.8,
    do_sample=True,
    repetition_penalty=1.1
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
### Input:
Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
### Response:
The chemicals present in metformin may act by blocking a certain enzyme called phosphodiesterase, which helps control blood sugar levels in the body. The enzyme is responsible for breaking down the sugars in our cells, so when it's blocked, this breaks down the sugars that are being stored in our bodies, allowing the sugar to leave the body through urination or sweating.
Because of its ability to break down the sugars in our cells


In [ ]:
model_path = "/content/tinyllama-preference-allignment/checkpoint-20"
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16)
preference_aligned_model.to("cuda")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.9,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Diabetes is a disease where your body produces too little or too much of insulin, which helps your cells to use glucose from food. Glucose is the main source of energy for the body, and if you don't make enough insulin, then not enough energy is made available for the body to function properly. The end result is that people with type 2 diabetes are usually at risk of developing heart disease, stroke, kidney failure,
